# Aykırı Değer Tespit Uygulama
Bu konuda, tek değişkenli aykırı değer tespiti için **iki temel yöntemi** 
(Z-Score ve IQR) pratik olarak uyguluyoruz — ikisini de daha önce farklı 
bağlamlarda görmüştük, şimdi doğrudan "aykırı değer 
avcılığı" amacıyla kullanıyoruz.

## Yöntem 1: Z-Score Yöntemi
$$Z = \frac{x - \bar{x}}{s}$$
**Kural:** |Z| > 3 (bazen 2 de kullanılır) olan gözlemler aykırı kabul 
edilir.

## Yöntem 2: IQR (Interquartile Range) Yöntemi
$$IQR = Q3 - Q1$$
**Kural:** 
- Alt sınır: $Q1 - 1.5 \times IQR$
- Üst sınır: $Q3 + 1.5 \times IQR$
- Bu sınırların dışında kalan her şey aykırı kabul edilir

**Z-Score'dan Farkı:** IQR, medyan/kartillere dayandığı için aykırı değerlere karşı Z-Score'dan daha **dayanıklıdır** — tıpkı medyanın ortalamaya göre dayanıklı olması gibi.

In [1]:
import numpy as np
import pandas as pd

np.random.seed(42)

# Normal bir veri + birkaç bilerek eklenmiş aykırı değer
veri = np.random.normal(50, 10, 100)
veri = np.append(veri, [120, 130, -20])  # 3 aykırı değer ekliyoruz

df = pd.DataFrame({'Deger': veri})

# --- Yöntem 1: Z-Score ---
df['Z_Score'] = (df['Deger'] - df['Deger'].mean()) / df['Deger'].std()
df['Aykiri_ZScore'] = df['Z_Score'].abs() > 3

print("Z-Score ile tespit edilen aykırı değerler:")
print(df[df['Aykiri_ZScore']])

# --- Yöntem 2: IQR ---
Q1 = df['Deger'].quantile(0.25)
Q3 = df['Deger'].quantile(0.75)
IQR = Q3 - Q1
alt_sinir = Q1 - 1.5 * IQR
ust_sinir = Q3 + 1.5 * IQR

df['Aykiri_IQR'] = (df['Deger'] < alt_sinir) | (df['Deger'] > ust_sinir)

print(f"\nIQR sınırları: [{alt_sinir:.2f}, {ust_sinir:.2f}]")
print("IQR ile tespit edilen aykırı değerler:")
print(df[df['Aykiri_IQR']])

# --- Karşılaştırma ---
print(f"\nZ-Score ile toplam aykırı: {df['Aykiri_ZScore'].sum()}")
print(f"IQR ile toplam aykırı: {df['Aykiri_IQR'].sum()}")

Z-Score ile tespit edilen aykırı değerler:
     Deger   Z_Score  Aykiri_ZScore
100  120.0  4.534579           True
101  130.0  5.180241           True
102  -20.0 -4.504682           True

IQR sınırları: [27.40, 71.64]
IQR ile tespit edilen aykırı değerler:
          Deger   Z_Score  Aykiri_ZScore  Aykiri_IQR
74    23.802549 -1.676520          False        True
100  120.000000  4.534579           True        True
101  130.000000  5.180241           True        True
102  -20.000000 -4.504682           True        True

Z-Score ile toplam aykırı: 3
IQR ile toplam aykırı: 4


### Sonuç
IQR tekniğinde sınırların daha dar olmasından dolayı 4 aykırı değer tespit edilirken, Z-Score yönteminde 3 tane aykırı değer tespit edildi. Her iki teknik de aykırı değer tespitinde sıkça kullanılan tekniklerdendir; IQR, medyan ve kartillere dayandığı için 
aykırı değerlere karşı daha dayanıklı (robust) bir yöntemdir, bu yüzden pratikte genellikle IQR tercih edilir.